# Notebook für Review "Historisiertes Gemeindeverzeichnis"

## Install/Import Modules

In [ ]:
%pip install -q ipywidgets==8.0.4 ipycytoscape networkx nbformat plotly
import ipycytoscape as cy
import networkx as nx
import pandas as pd
import plotly.express as px
from ext.sparql import query, display_result

In [ ]:
async def show_meta(URI, store):
    
    df = await query("""
    
    SELECT ?s1 ?p1 ?truc ?p2 ?o2
    FROM <https://lindas.admin.ch/fso/register>
    WHERE {

        BIND(<""" + URI + """> AS ?truc)
        {
            ?s1 ?p1 ?truc.
        }
        UNION
        {
            ?truc ?p2 ?o2.
        }

    }

    """, store)
    
    display_result(df)

# Datenmodell

## version.link

Die Daten zu den Gemeinden sind gemäss dem Schema https://version.link transformiert und in den Namespaces ld.admin.ch/municipality/ resp. ld.admin.ch/municipality/version/ zu finden.

## register.ld.admin.ch/agvch

Die Ursprungsdaten zum historisierten Gemeindeverzeichnis werden vom BFS als [eCH-0071 Standard](https://www.ech.ch/de/ech/ech-0071/1.1-0) gepflegt. Als direkte Linked Data Konversion existieren die Daten im Namespace register.ld.admin.ch/agvch/municipality/ resp. register.ld.admin.ch/agvch/municipalityversion/. 

# vl:Identity 

## Beispiel aktuelle Gemeinde (Bern)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/351> ?p ?o.

}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Mandatory Properties aus version.link für eine Identität sind:

- OK: rdf:type vl:Identity
- OK: vl:version
- OK: vl:inVersionedIdentitySet
- OK: schema:identifier

Gemeinde unter register.ld.admin.ch/agvch/municipality/

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://register.ld.admin.ch/agvch/municipality/351> ?p ?o.

}


""", "https://test.lindas.admin.ch/query")

display_result(df)

## Beispiel nicht mehr aktive Gemeinde (Ennenda)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/1607> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Mandatory Properties aus version.link für eine nicht mehr existierende Identität sind:

- OK: rdf:type vl:Identity
- OK: vl:version
- OK: vl:inVersionedIdentitySet
- OK: schema:identifier
- OK: vl:Deprecated

--> Frage: Für was ist das schema:DefinedTermSet https://ld.admin.ch/dimension/municipality notwendig? Wohl für visualize.admin.ch - aber gehören da vl:Deprecated auch rein?

## Beispiel aktueller Bezirk (Bern-Mittelland)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/district/246> ?p ?o.

}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Mandatory Properties aus version.link für eine Identität sind:

- OK: rdf:type vl:Identity
- OK: vl:version
- **NOK**: vl:inVersionedIdentitySet --> ist zwar vorhanden, sollte aber für das gesamte Set AGVCH das gleiche vl:IdentitySet sein
- OK: schema:identifier

## Beispiel nicht mehr aktiver Bezirk (Arbon)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/district/2001> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

## Beispiel Kanton (Bern)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    <https://ld.admin.ch/canton/2> ?p ?o.
}

""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Mandatory Properties aus version.link für eine Identität sind:

- OK: rdf:type vl:Identity
- **NOK**: vl:version
- **NOK**: vl:inVersionedIdentitySet
- OK: schema:identifier

## Hinweis zu vl:Deprecated

version.link lässt offen, dass man aktiven Gemeinden bspw. admin:Municipality und nicht mehr aktiven admin:AbolishedMunicipality zuweist (und kein admin:Municipality mehr), somit könnte man einfacher nach allen aktiven suchen. Der Grund wieso ein vl:Deprecated eingeführt wurde, ist vor allem für die Versionen: Das Prinzip dort ist, dass man nichts ändern will, was man mal hinzugefügt hat, man will also nicht, dass bspw. ein vl:ActiveVersion durch ein vl:DeprecatedVersion ersetzt wird, dieser Graph ist "strictly growing", man muss also ein vl:Deprecated hinzutun, ohne etwas wegzunehmen. Damit bei den Identities der gleiche Mechanismus spielt, ist es dort auch so, aber in der Praxis könnte man immer zusätzlich zu vl:Identity als Typ auch etwas setzen, das man dann wenn die Identität abolished wird, weglässt und durch ein Spezifikum ersetzt, also bspw. admin:PoliticalMunicipality durch admin:AbolishedPoliticalMunicipality

## Anzahl aller Gemeinden

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

## Anzahl aller nicht mehr aktiven Gemeinden

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Deprecated.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

## Anzahl aller aktuellen Gemeinden

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
    
    FILTER NOT EXISTS {?muni a vl:Deprecated}
    #MINUS {?muni a vl:Deprecated}

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

## Anzahl aktueller Gemeinden pro Kanton

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?canton ?cantonName ?numberOfMunies
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                schema:isPartOf/schema:isPartOf ?canton.

            MINUS {?muni a vl:Deprecated}

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfMunies)

""", "https://test.lindas.admin.ch/query")

display_result(df)

## Anzahl nicht mehr aktiver Gemeinden pro Kanton

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?cantonName ?numberOfMunies
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                schema:isPartOf/schema:isPartOf ?canton;
                a vl:Deprecated.

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfMunies)

""", "https://test.lindas.admin.ch/query")

display_result(df)

## Anzahl aktueller Bezirke pro Kanton 

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?cantonName ?numberOfDistricts
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?district) AS ?numberOfDistricts)
        WHERE {

            ?district a admin:District;
                schema:isPartOf ?canton.

            MINUS {?district a vl:Deprecated}

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfDistricts)

""", "https://test.lindas.admin.ch/query")

display_result(df)

# vl:Version

## Beispiel für eine Version einer Gemeinde (Ennenda)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/version/12279> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Mandatory Properties aus version.link für eine Version sind:

- OK: rdf:type vl:Version
- OK: vl:identity
- OK: vl:identityIdentifier
- **NOK**: vl:inVersionedIdentitySet --> ist zwar vorhanden, sollte aber für das gesamte Set AGVCH das gleiche vl:IdentitySet sein
- OK: schema:identifier
- OK: vl:successor

**Die Version der Gemeinde zeigt auf eine Version des Districts, version.link verlangt hier aber eigentlich der Link auf die Identität des Bezirks** --> nochmals darüber nachdenken

Wenn es so gemacht wird wie aktuell (und nicht wie in version.link vorgesehen), heisst das, dass es Versionen von Gemeinden gibt, die nicht mehr aktuell sind, die aber auf eine Version eines Districts zeigen, der noch aktuell ist...

Version unter register.ld.admin.ch/agvch/municipalityversion/

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://register.ld.admin.ch/agvch/municipalityversion/12279> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

## Beispiel für eine Version eines Bezirks (Arbon)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/district/version/10001> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Mandatory Properties aus version.link für eine Version sind:

- OK: rdf:type vl:Version
- OK: vl:identity
- OK: vl:identityIdentifier
- **NOK**: vl:inVersionedIdentitySet --> ist zwar vorhanden, sollte aber für das gesamte Set AGVCH das gleiche vl:IdentitySet sein
- OK: schema:identifier
- OK: vl:successor

## Beispiel für eine Version eines Sees (Walensee)

Achtung, es gibt Versionen von Gemeinden, die eigentlich ein See sind, die dazugehörige Identität existiert bist jetzt noch nicht:

In [ ]:
await show_meta("https://ld.admin.ch/municipality/version/10254", "https://test.ld.admin.ch/query")

In [ ]:
await show_meta("https://ld.admin.ch/municipality/9268", "https://test.ld.admin.ch/query")

## Versionen, die keine Gemeinden und keine Bezirke sind

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version a vl:Version;
        vl:identity ?identity;
        schema:name ?name.
    
    FILTER NOT EXISTS {?identity a admin:Municipality}
    FILTER NOT EXISTS {?identity a admin:District}


} ORDER BY ?name

""", "https://test.lindas.admin.ch/query")

display_result(df)

## Test vl:Deprecated auf letzter Version

Laut Anforderungen von version.link, muss im Falle, dass eine Identität vl:Deprecated erhält, die letzte Version ebenfall ein vl:Deprecated haben:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Deprecated;
        vl:version ?version.
    ?version a vl:Deprecated.
        
    #MINUS {?version a vl:Deprecated}

} LIMIT 10

""", "https://test.lindas.admin.ch/query")

display_result(df)

Scheinbar haben diese Versionen alle nicht ein vl:Deprecated..., gibt es überhaupt vl:Version mit vl:Deprecated:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version a vl:Version;
        vl:identity ?muni;
        a vl:Deprecated.
    ?muni a admin:PoliticalMunicipality

} LIMIT 10

""", "https://test.lindas.admin.ch/query")

display_result(df)

es gibt keine Versionen mit vl:Deprecated

## Gemeinden zu einem bestimmten Zeitpunkt in einem Kanton

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?date ?muniVersion ?name
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    BIND ("2000-01-01"^^xsd:date AS ?date)
    
    ?muniVersion a vl:Version;
        vl:identity ?muni;
        schema:isPartOf/schema:isPartOf <https://ld.admin.ch/canton/8>;
        schema:startDate ?start;
        schema:legalName ?name.
    ?muni a admin:PoliticalMunicipality.
        
    OPTIONAL {?muniVersion schema:endDate ?stop.}
    
    BIND (IF(BOUND(?stop), ?stop, ?date) AS ?stop2)
    
    FILTER (?date >= ?start)
    FILTER (?date <= ?stop2)

}

""", "https://test.ld.admin.ch/query")

display_result(df)

## Lückenlose Versionen

Gibt es Gemeinden, die lückenhafte Versionen haben 

## Wiederauferstandene Gemeinden

Gibt es Gemeinden, deren Identität nicht lückenlos bestanden hat (Wiederauferstehung):

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    {
        SELECT ?muniIdentity (COUNT(?startEvent) AS ?count)
        WHERE {

            ?muniIdentity a vl:Identity;
                a admin:Municipality.
            ?muniVersion vl:identity ?muniIdentity;
                vl:startEvent ?startEvent.
            ?startEvent a vl:InitialRecording.
        } GROUP BY ?muniIdentity
    }
    FILTER(?count != 1)
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Es gibt keine wiederauferstandenen Gemeinden, also zumindest nicht mit mehreren vl:InitialRecording. Etwas anders sieht es aus, wenn das keine Anforderung für Wiederauferstehung ist, siehe [hier](#Zeitstrahl-von-Versionen-(Berg-(TG))).

# Veränderungsprozesse

## Beispiel für Versionsgeschichte Fusion (Klosters)

Welche Gemeinden spielten alle eine Rolle bei der heutigen Gemeinde "Klosters" (BFS Nummer 3871)?

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?participants ?name
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/3871> vl:version ?final.
    ?participants vl:successor* ?final;
        schema:name ?name.
    
    MINUS {?participants a vl:ChangeEvent}
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Grafische Darstellung

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/3871> vl:version ?final.
    ?source vl:successor* ?final;
        vl:successor ?target.
    
    MINUS {?source a vl:ChangeEvent}
}


""", "https://test.lindas.admin.ch/query")


G = nx.from_pandas_edgelist(df, create_using=nx.DiGraph())

my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'label': 'data(id)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

## Beispiel für Versionsgeschichte Teilung (Bolligen)

Für Bolligen (BFS Nummer 352) als Beispiel für eine Gemeinde, die sich geteilt hat:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/352> vl:version ?final.
    ?source vl:successor* ?final;
        vl:successor ?target;
        schema:name ?sourceName.
    ?target schema:name ?targetName.
    
    MINUS {?source a vl:ChangeEvent}
}


""", "https://test.lindas.admin.ch/query")


G = nx.from_pandas_edgelist(df, source="source", target="target", create_using=nx.DiGraph())

my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'label': 'data(id)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

## Alle Gemeinden, die sich geteilt haben

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?source schema:name ?name
    
    {
        SELECT ?source (COUNT(?target) as ?successors)
        WHERE {

            ?source vl:successor ?target;

            MINUS {?source a vl:ChangeEvent}
            
            FILTER(regex(str(?source), "municipality" ) )
        } 
        GROUP BY ?source
    }
    FILTER(?successors > 1)
}

""", "https://test.lindas.admin.ch/query")

display_result(df)

Problem: Wenn man nicht dir URI filtert, sind da auch Districts dabei und nicht nur Municipalities, es ist schwierig, nur nach Versionen von Gemeinden zu suchen --> sollte man wohl noch was anhängen admin:MunicipalityVersion

## Analyse Berg (TG)

Dies ist ein Beispiel für eine Abfrage, wo zu Beginn oder am Ende nicht eine einzelne Version steht, sondern man von einer Version in der "Mitte" des Geschehens ausgeht. Zusätzlich sollen nicht mehr nur die URI der Versionen in der Grafik angezeigt werden, sondern die Namen, dafür muss man Node Attribute generieren (man kann nicht einfach nur auf die Namen setzten, weil die sind ja nicht eindeutig).

In [ ]:
edge_df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target ?sourceName ?targetName
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    {
        <https://ld.admin.ch/municipality/version/14062> vl:predecessor* ?target.
        ?target vl:predecessor ?source;
            schema:name ?targetName.
        ?source schema:name ?sourceName.
    }
    UNION
    {
        <https://ld.admin.ch/municipality/version/14062> vl:successor* ?source.
        ?source vl:successor ?target;
            schema:name ?sourceName.
        ?target schema:name ?targetName.
    }
}

""", "https://test.lindas.admin.ch/query")

ids = pd.Series.tolist(edge_df["source"]) + pd.Series.tolist(edge_df["target"])
names = pd.Series.tolist(edge_df["sourceName"]) + pd.Series.tolist(edge_df["targetName"])

node_df = pd.DataFrame(list(zip(ids, names)), columns = ["id", "name"])
node_df.drop_duplicates(inplace=True)

G = nx.from_pandas_edgelist(edge_df, source="source", target="target", create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))


my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '12px',
            'label': 'data(name)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

## Zeitstrahl von Versionen (Berg (TG))

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?version ?Start ?Stop2
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version vl:identity <https://ld.admin.ch/municipality/4891>;
        schema:startDate ?Start.
        OPTIONAL {?version schema:endDate ?Stop}
        
        BIND ("2023-04-01"^^xsd:date AS ?today).
        
        BIND (IF(BOUND(?Stop), ?Stop, ?today) AS ?Stop2).
    
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

fig = px.timeline(df, x_start="Start", x_end="Stop2", y="version")
fig.update_yaxes(autorange="reversed") # otherwise tasks are listed from the bottom up
fig.show()

Interessante Geschichte: Einerseits gibt es eine Lücke in der Identität nach 11761, andererseits hat man eine Version, die nur an einem Tag existiert (14062).

## Versionen, die ohne Zeitausdehnung existieren

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version a vl:Version;
        schema:name ?name;
        schema:startDate ?start;
        schema:endDate ?stop.
        
    FILTER(?start = ?stop)
    
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?predecessor ?predecessorAbolitionMode ?admissionMode ?version ?abolitionMode ?successorAdmissionMode ?successor
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version a vl:Version;
        schema:name ?name;
        schema:startDate ?start;
        schema:endDate ?stop;
        vl:predecessor ?predecessor;
        vl:successor ?successor.
        
    ?version <http://www.w3.org/ns/prov#hadPrimarySource> ?versionAGVCH.
    ?predecessor <http://www.w3.org/ns/prov#hadPrimarySource> ?predecessorAGVCH.
    ?successor <http://www.w3.org/ns/prov#hadPrimarySource> ?successorAGVCH.
            
    ?predecessorAGVCH <https://ld.admin.ch/ech/71/municipalityAbolitionModeId> ?predecessorAbolitionMode.
    ?versionAGVCH <https://ld.admin.ch/ech/71/municipalityAdmissionModeId> ?admissionMode.
    ?versionAGVCH <https://ld.admin.ch/ech/71/municipalityAbolitionModeId> ?abolitionMode.
    ?successorAGVCH <https://ld.admin.ch/ech/71/municipalityAdmissionModeId> ?successorAdmissionMode.

    #MINUS {?predecessor a vl:ChangeEvent}
    #MINUS {?successor a vl:ChangeEvent}
        
    FILTER(?start = ?stop)
    
} ORDER BY ?version


""", "https://test.lindas.admin.ch/query")

display_result(df)

# vl:ChangeEvent

## Beispiel Change Event

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/changeevent/3953> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Problem: Es ist schwierig rauszufinden, welcher Change Typ ein ChangeEvent aufweist, weil das alles als rdf:type ausgeführt ist. Gemäss version.link sollte der Change Typ ja eine Subklasse von vl:ChangeEvent sein, aber wie kann man das in LINDAS abfragen?

Wäre es allenfalls sinnvoller, den Change Typ mit vl:changeEventType an den vl:ChangeEvent anzuhängen?

## Abfrage Change Typ 

Welchen Change Typ hat ein bestimmtes vl:ChangeEvent (ohne die anderen Klassen mitzuhaben):

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/changeevent/3953> a ?type.
    
    FILTER(?type != vl:ChangeEvent)
    FILTER(?type != admin:MunicipalityChangeEvent)
    FILTER(?type != <https://ld.admin.ch/ech/71/MunicipalityChangeEvent>)
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

## Verwendete Change Typen

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?type (COUNT(?change) as ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?change a vl:ChangeEvent;
        a ?type.
    
    FILTER(?type != vl:ChangeEvent)
    FILTER(?type != admin:MunicipalityChangeEvent)
    FILTER(?type != <https://ld.admin.ch/ech/71/MunicipalityChangeEvent>)
} GROUP BY ?type 


""", "https://test.lindas.admin.ch/query")

display_result(df)

Wo sind die vl:Combination etc.?

## Art der Changes (Beispiel Klosters)

Jetzt soll nicht nur der Successor resp. der Predecessor einer Version eine Rolle spielen, sondern der Prozess, der zu dieser Version geführt hat:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?predecessor ?changeEvent ?type ?successor
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/3871> vl:version ?final.
    ?predecessor vl:successor* ?final;
        vl:successor ?successor;
        vl:endEvent ?changeEvent.

    ?changeEvent a ?type.
    
    #FILTER(?type != vl:ChangeEvent)
    #FILTER(?type != admin:MunicipalityChangeEvent)
    #FILTER(?type != <https://ld.admin.ch/ech/71/MunicipalityChangeEvent>)
    
    

} ORDER BY ?predecessor


""", "https://test.lindas.admin.ch/query")

display_result(df)

Problem: vl:ChangeEvent 3450 hat kein Change Typ, müsste vl:Combination sein

## Art der Changes aus den AGVCH Daten (Beispiel Klosters)

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?predecessor ?abolitionMode ?admissionMode ?successor
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/3871> vl:version ?final.
    ?predecessor vl:successor* ?final;
        vl:successor ?successor;
        <http://www.w3.org/ns/prov#hadPrimarySource> ?predecessorAGVCH.
    ?successor <http://www.w3.org/ns/prov#hadPrimarySource> ?successorAGVCH.
        
    ?predecessorAGVCH <https://ld.admin.ch/ech/71/municipalityAbolitionModeId> ?abolitionMode.
    ?successorAGVCH <https://ld.admin.ch/ech/71/municipalityAdmissionModeId> ?admissionMode.
        
    

    MINUS {?predecessor a vl:ChangeEvent}
    

} ORDER BY ?predecessor


""", "https://test.lindas.admin.ch/query")

display_result(df)

Es ist also immer möglich, die "Ursprungsdaten" wie im BFS XML zur Verfügung gestellt, auch auszuwerten.

# Hierarchielevel

## Nicht mehr aktuelle Versionen von Gemeinden zeigen auf aktuelle Versionen von Bezirken

Im Moment verlinken Versionen von Gemeinden zu Versionen von Bezirken. Damit man keine Kaskaden kreiert, muss es die Möglichkeit geben, dass nicht mehr aktuelle Versionen von Gemeinden immer noch auf aktuelle Versionen von Bezirken zeigen:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?muniVersion ?muniVersionEnd ?districtVersionEnd ?districtVersion
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muniVersion a vl:Version;
        vl:identity ?muniIdentity;
        schema:isPartOf ?districtVersion.
    ?muniIdentity a admin:PoliticalMunicipality.
        
    OPTIONAL {?muniVersion schema:endDate ?muniVersionEnd.}

    OPTIONAL {?districtVersion schema:endDate ?districtVersionEnd.}
    
} LIMIT 100


""", "https://test.lindas.admin.ch/query")

display_result(df)

## Alle aktuellen Gemeinde-Versionen einer aktuellen Bezirks-Version

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?muniVersion ?muniVersionEnd ?districtVersionEnd ?districtVersion
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    BIND(<https://ld.admin.ch/district/version/10107> as ?districtVersion)
    
    ?muniVersion a vl:Version;
        vl:identity ?muniIdentity;
        schema:isPartOf ?districtVersion.
        
    OPTIONAL {?muniVersion schema:endDate ?muniVersionEnd.}

    OPTIONAL {?districtVersion schema:endDate ?districtVersionEnd.}
    
} LIMIT 100


""", "https://test.lindas.admin.ch/query")

display_result(df)

## n:1 Beziehungen

Aktuell ist es so gelöst (auch im XML vom BFS), dass eine Gemeindeversion nur zu einer Bezirksversion gehören kann, wenn die Bezirke ändern, hat das auch eine neue Gemeindeversion zur Folge, weil die bestehende Gemeindeversion sonst ja auf mehrere Bezirksversionen zeigen müsste:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
    
SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    {
        SELECT ?muniVersion (COUNT(?districtVersion) AS ?count)
        WHERE {

            ?muniVersion a vl:Version;
                schema:isPartOf ?districtVersion;
                vl:identity ?muniIdentity.
            ?muniIdentity a admin:Municipality.


        } GROUP BY ?muniVersion
        
    }
    
    FILTER(?count > 1)
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Dass keine Resultate erzielt werden, zeigt, dass es keine Gemeindeversionen gibt, die auf mehrere Districtversionen zeigen.

# Verschachtelte vl:ChangeEvent

Der Abschnitt https://version.link/#ChangeEvent ist wohl noch zu wenig ausführlich geschrieben. Hier ein Beispiel, wenn man zwei Gemeinden fusionieren lässt, wobei es für die eine Gemeinde eine Aufhebung ist, für die andere eine Gebietsänderung (kleine Gemeinde fusioniert in grosse):

Es sind also drei Versionen beteiligt:

- VersionKleinVorher
- VersionGrossVorher
- VersionGrossNachher

Weiter sind auch drei Changes beteiligt:

- vl:Combination
- Abolishment (nicht in version.link definiert)
- Reshape (nicht in version.link definiert)

Der Event vl:Combination verbindet als vl:endEvent die Versionen *VersionKleinVorher* und *VersionGrossVorher* über vl:startEvent mit der Version *VersionGrossNachher*.

Der Event Abolishment verbindet nur die Versionen *VersionKleinVorher* mit *VersionGrossNachher und der Event Reshape nur *VersionGrossVorher* mit *VersionGrossNachher*.

Frage: Würde man das evt. nicht besser als irgendwie vl:SubChangeEvent innerhalb des vl:Combination abhandeln, weil sonst hat man das Problem, dass *VersionKleinVorher* drei vl:successor hat: *VersionGrossNachher*, vl:Combination und Abolishment, aber vielleicht muss das historisierte Gemeindeverzeichnis gar nicht so weit gehen, weil die Details hätte man ja dann noch im register.ld.admin.ch/agvh/municipalityversion...